[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C53_RealTime_Detectors_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与实时检测的三本账（延迟预算 / NMS 波动 / 帕累托前沿）

本课全程 **纯 numpy + 标准库、CPU、不联网**。这个 notebook 不训练任何模型，
它建立的是**三本你在后面五个模块里会反复用到的账**。

**本 notebook 你会亲手实现：**
1. **帧延迟预算分解器** —— 从 33.3 ms 帧周期一路算到 TSR 分到的 2.45 ms，再拆到六个阶段
2. **NMS 的目标数依赖模型** —— 量化「唯一随场景波动的那一项」，看它怎么在雨夜吃掉整个预算
3. **帕累托前沿与「被支配」判定** —— 11 个模型里划掉 5 个，并给出「谁支配了它」
4. **延迟-精度曲线的正确读法** —— 同延迟比 AP，以及为什么反过来是错的
5. **TSR 的物理约束** —— 针孔模型算出 60 m 外的限速牌只有 16.6 px，以及 IoU 对位移的敏感性

> 心智模型：**延迟预算与显存是外部硬约束，精度是唯一的优化目标。
> 实时检测的整部历史 = 在给定的毫秒数里把 AP 往上抬。**

## 1 · 环境自检

In [ ]:
import sys, platform, math, json, itertools, time
print('Python', sys.version.split()[0], '|', platform.system(), platform.machine())
import numpy as np
print('numpy', np.__version__)
for name in ['torch', 'torchvision', 'onnxruntime', 'cv2']:
    try:
        mod = __import__(name)
        print('  %-14s %-10s (有更好，但本课全程不需要)' % (name, getattr(mod, '__version__', '?')))
    except ImportError:
        print('  %-14s %s' % (name, '未安装 -> 走纯 numpy 复现路径'))

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
rng = np.random.default_rng(0)
print('\n✅ 环境就绪：本课 CPU 可跑、不联网、不下载数据集。')

## 2 · 第一本账：33.3 ms 到底怎么花的

「实时」不是形容词。把它换成数字，很多设计选择立刻有了解释。

**注意这里的两级瓜分**：先被固定开销切一刀，剩下的时间窗再被十几个感知任务分。

In [ ]:
FPS = 30
FRAME_MS = 1000.0 / FPS

# 第一级：固定开销（TSR 代码改不动的部分）
FIXED = [
    ('传感器曝光 + ISP + MIPI 传输', 8.0, '相机侧固定开销'),
    ('融合/跟踪/地图匹配/规控/执行下发', 7.0, 'TSR 的下游，不能挤压'),
    ('调度抖动与安全裕度', 2.0, '车端看 p99，不是 p50'),
]
fixed_ms = sum(x[1] for x in FIXED)
perception_ms = FRAME_MS - fixed_ms

# 第二级：全部感知任务瓜分剩下的时间窗
TASK_SHARE = [
    ('BEV 障碍物检测', 0.45), ('车道线 / 可行驶区域', 0.20),
    ('交通标志 TSR', 0.15), ('红绿灯', 0.10), ('其他（静态障碍/占位）', 0.10),
]
assert abs(sum(s for _, s in TASK_SHARE) - 1.0) < 1e-9

print('帧周期 @ %d FPS = %.2f ms' % (FPS, FRAME_MS))
for name, ms, why in FIXED:
    print('   - %-36s %5.2f ms   %s' % (name, ms, why))
print('   = 留给「全部感知任务」的时间窗: %.2f ms\n' % perception_ms)

tsr_budget = None
for name, share in TASK_SHARE:
    ms = perception_ms * share
    mark = '   <-- 本课的主角' if name.startswith('交通标志') else ''
    print('     %-24s %3.0f%%   %5.2f ms%s' % (name, share * 100, ms, mark))
    if name.startswith('交通标志'):
        tsr_budget = ms

assert abs(FRAME_MS - 33.3333) < 1e-3
assert abs(perception_ms - 16.3333) < 1e-3
assert 2.4 < tsr_budget < 2.5, tsr_budget
print('\n>>> TSR 的端到端延迟预算 = %.2f ms' % tsr_budget)
print('    这个数字在后面每一个模块里都会被拿出来对照。')

In [ ]:
# 把 2.45 ms 拆到阶段级 —— 这一层才是你能动的
STAGES = [
    ('预处理 resize+letterbox+归一化', 0.35, '常在 CPU；训练-部署不一致的头号来源(C60)'),
    ('H2D 拷贝',                       0.15, '可与推理重叠'),
    ('网络前向 backbone+neck+head',    1.45, '唯一能靠换模型优化的一项'),
    ('NMS',                            0.30, '随目标数波动 —— 唯一不确定的一项'),
    ('解码/坐标还原/阈值',             0.12, '写错就是「框整体偏移」(C60)'),
    ('D2H 拷贝',                       0.05, ''),
]
used = sum(x[1] for x in STAGES)

print('%-36s %8s %8s   %s' % ('阶段', '毫秒', '占比', '备注'))
for name, ms, note in STAGES:
    print('%-36s %8.2f %7.1f%%   %s' % (name, ms, 100 * ms / used, note))
print('%-36s %8.2f            预算 %.2f ms，余量 %.2f ms'
      % ('合计', used, tsr_budget, tsr_budget - used))

infer_share = 1.45 / used
assert used <= tsr_budget, '端到端超预算'
assert 0.55 < infer_share < 0.65, infer_share
print('\n⚠️  网络前向只占 %.0f%%。「换个快一倍的 backbone」对端到端最多带来 %.0f%% 的收益。'
      % (100 * infer_share, 100 * infer_share / 2))
print('    另外 %.0f%% 在预处理/拷贝/NMS/后处理里 —— 而它们几乎不出现在论文的 FPS 表里。'
      % (100 * (1 - infer_share)))

## 3 · 第二本账：NMS 是唯一「不确定」的一项

其余五项的耗时只和输入分辨率有关（给定模型后是常数）。
**只有 NMS 的耗时随场景变化**——因为它取决于过了分数阈值的候选框数量。

这就是 RT-DETR 存在的直接理由（模块 04 会展开）。

In [ ]:
def nms_cost_ms(n, base=0.05, per_pair=2.2e-6):
    '''朴素 NMS 的代价模型：固定开销 + 成对 IoU 计算。
       真实实现有向量化与早停，常数不同，但 O(n^2) 的形状是一样的。'''
    return base + per_pair * n * (n - 1) / 2

print('%12s %14s %16s' % ('候选框数 n', 'NMS 耗时(ms)', '占 TSR 预算'))
for n in [50, 100, 300, 1000, 3000, 10000]:
    c = nms_cost_ms(n)
    print('%12d %14.3f %15.0f%%' % (n, c, 100 * c / tsr_budget))

assert nms_cost_ms(100) < 0.3
assert nms_cost_ms(3000) > tsr_budget, 'n=3000 时 NMS 单项就该爆掉整个预算'

# 工程上的止血法：NMS 前先做 top-k 预筛
def nms_cost_with_topk(n, topk=300):
    return nms_cost_ms(min(n, topk))

print('\ntop-k 预筛（topk=300）的效果：')
for n in [300, 1000, 3000, 10000]:
    print('   n=%-6d 无预筛 %9.3f ms   ->  预筛后 %6.3f ms'
          % (n, nms_cost_ms(n), nms_cost_with_topk(n)))
assert nms_cost_with_topk(10000) == nms_cost_ms(300)
print('✅ top-k 把**最坏情况**封住了，但没有消除方差 —— 见下一格。')

In [ ]:
# 同一个模型，不同场景下的端到端延迟
SCENES = [('空旷高速', 40), ('城市主干道', 120), ('施工区门架(一排标志)', 260),
          ('雨夜 + 广告牌误检爆炸', 900)]
base_ms = used - 0.30                       # 扣掉表里那个「典型 NMS」

print('%-26s %10s %14s %10s' % ('场景', '候选框数', '端到端(ms)', '是否超预算'))
lat = {}
for name, n in SCENES:
    t = base_ms + nms_cost_ms(n)
    lat[name] = t
    print('%-26s %10d %14.3f %10s' % (name, n, t, '❌ 超' if t > tsr_budget else '✅'))

p50, p99 = lat['城市主干道'], lat['雨夜 + 广告牌误检爆炸']
print('\np50 ≈ %.2f ms   p99 ≈ %.2f ms   ->  尾延迟是中位数的 %.2f 倍'
      % (p50, p99, p99 / p50))
assert lat['空旷高速'] < tsr_budget < lat['雨夜 + 广告牌误检爆炸']
assert p99 / p50 > 1.3
print('\n⚠️  **车端安全关心的是 p99，不是均值**。一个「平均 2.2 ms」的检测器')
print('    如果 p99 是 3.1 ms，它就是超预算的 —— 而论文只会报平均。')
print('✅ 这就是「NMS 是延迟不确定性的唯一来源」这句话的定量含义（模块 04 会重做这个实验）。')

## 4 · 第三本账：帕累托前沿与「被支配」判定

模型 A **支配** 模型 B，当且仅当 A 不比 B 慢、不比 B 差，且至少有一项严格更好。
被支配的模型可以直接从候选名单划掉——**存在一个又快又准的替代品**。

> ⚠️ 下表的数字取自各论文自报的量级（不同论文测法不同，模块 05 会拆穿这件事）。
> 这里只用来演示**读图的方法**，不要当成选型结论。

In [ ]:
# (名字, 延迟 ms, COCO AP)  —— 量级示意
MODELS = [
    ('YOLOv5-S',      4.5, 37.4), ('YOLOv8-S',      5.1, 44.9),
    ('RTMDet-S',      5.4, 44.5), ('RT-DETR-R18',   6.8, 46.5),
    ('YOLOv5-M',      7.0, 45.4), ('YOLOv8-M',      8.2, 50.2),
    ('RT-DETR-R50',   9.3, 53.1), ('YOLOv5-L',     10.1, 49.0),
    ('RTMDet-L',     10.2, 51.3), ('YOLOv8-L',     12.8, 52.9),
    ('RT-DETR-R101', 13.5, 54.3),
]

def dominates(a, b):
    '''a 支配 b: 不更慢、不更差，且至少一项严格更好。'''
    return a[1] <= b[1] and a[2] >= b[2] and (a[1] < b[1] or a[2] > b[2])

def pareto_front(models):
    out = []
    for i, m in enumerate(models):
        if not any(dominates(o, m) for j, o in enumerate(models) if j != i):
            out.append(m)
    return sorted(out, key=lambda x: x[1])

def dominators(name, models):
    m = [x for x in models if x[0] == name][0]
    return [o[0] for o in models if o[0] != name and dominates(o, m)]

front = pareto_front(MODELS)
names = [m[0] for m in front]
print('帕累托前沿（%d / %d 个模型）：' % (len(front), len(MODELS)))
for nm, t, ap in front:
    print('   %-14s %5.1f ms   AP %.1f' % (nm, t, ap))

print('\n被支配的模型（可以直接划掉）：')
for nm, t, ap in sorted(MODELS, key=lambda x: x[1]):
    d = dominators(nm, MODELS)
    if d:
        print('   %-14s %5.1f ms AP %.1f   被 %s 支配（更快且更准）' % (nm, t, ap, ' / '.join(d)))

assert len(front) == 6, front
assert 'RTMDet-S' not in names and 'YOLOv5-L' not in names
assert set(dominators('YOLOv5-L', MODELS)) == {'YOLOv8-M', 'RT-DETR-R50'}
print('\n✅ 11 个模型里 5 个被支配。**选型的第一步永远是先把被支配的划掉。**')

## 5 · 曲线的正确读法：同延迟比 AP

In [ ]:
def best_under_budget(models, budget_ms):
    ok = [m for m in models if m[1] <= budget_ms]
    return max(ok, key=lambda m: m[2]) if ok else None

print('%12s   %s' % ('预算(ms)', '预算内 AP 最高的模型'))
for b in [tsr_budget, 5.0, 6.0, 9.0, 10.0, 13.0, 14.0]:
    r = best_under_budget(MODELS, b)
    txt = ('%-14s AP %.1f' % (r[0], r[2])) if r else '❌ 没有任何模型放得进来'
    print('%12.2f   %s' % (b, txt))

assert best_under_budget(MODELS, 9.0)[0] == 'YOLOv8-M'
assert best_under_budget(MODELS, 10.0)[0] == 'RT-DETR-R50'
assert best_under_budget(MODELS, tsr_budget) is None

gain = best_under_budget(MODELS, 10.0)[2] - best_under_budget(MODELS, 9.0)[2]
assert abs(gain - 2.9) < 1e-6
print('\n✅ 正确读法：「预算从 9.0 放宽到 10.0 ms，可拿到的 AP 从 50.2 涨到 53.1（+%.1f）」' % gain)
print('   —— 这句话可以直接拿去和产品谈判：多给我 1 ms，我给你 2.9 AP。')

d = dict((m[0], m) for m in MODELS)
print('\n❌ 错误读法：「RT-DETR-R101 比 YOLOv8-M 高 %.1f AP」'
      % (d['RT-DETR-R101'][2] - d['YOLOv8-M'][2]))
print('   —— 它同时慢了 %.0f%%（%.1f -> %.1f ms）。跨预算档比 AP 没有选型价值，'
      % (100 * (d['RT-DETR-R101'][1] / d['YOLOv8-M'][1] - 1), d['YOLOv8-M'][1], d['RT-DETR-R101'][1]))
print('      而这是论文里最常见的一句话。')
print('\n❗ 而 TSR 的 %.2f ms 预算里**一个模型都放不进来** —— 这不是账算错了。' % tsr_budget)
print('   真实工程的三条出路：① tiny 档 + INT8 + 更小输入（代价：小目标掉点，C57）')
print('                     ② TSR 降频到 10-15 Hz（代价：首检距离变远，C55）')
print('                     ③ 只在 ROI/消失点附近跑（代价：漏掉侧向标志）')

## 6 · TSR 的物理约束：60 米外的限速牌只有 16.6 像素

这门课反复回到交通标志检测，因为它同时踩中了实时检测最难的两点：
**极小的目标** + **极紧的延迟预算**。先把「极小」变成数字。

针孔相机模型：焦距（像素）$f = \frac{W/2}{\tan(\mathrm{HFOV}/2)}$，
物体成像边长 $px = f \cdot S / Z$（$S$ 物理尺寸，$Z$ 距离）。

In [ ]:
def focal_px(width_px, hfov_deg):
    '''横向分辨率 + 水平 FOV -> 焦距（像素）'''
    return (width_px / 2.0) / math.tan(math.radians(hfov_deg) / 2.0)

def sign_px(size_m, dist_m, f_px):
    return f_px * size_m / dist_m

CAMS = [('前视主摄 1920x1080 / 60°', 1920, 60.0),
        ('前视长焦 1920x1080 / 30°', 1920, 30.0),
        ('高分主摄 3840x2160 / 60°', 3840, 60.0)]

print('直径 0.6 m 的圆形限速牌，成像边长（像素）：')
print('%-28s %9s %9s %9s %9s' % ('相机', '30 m', '60 m', '100 m', '150 m'))
for name, wpx, fov in CAMS:
    f = focal_px(wpx, fov)
    row = [sign_px(0.6, z, f) for z in (30, 60, 100, 150)]
    print('%-28s %9.1f %9.1f %9.1f %9.1f' % (name, row[0], row[1], row[2], row[3]))

f_main = focal_px(1920, 60.0)
px60, px100 = sign_px(0.6, 60, f_main), sign_px(0.6, 100, f_main)
assert 1660 < f_main < 1665, f_main
assert 16.0 < px60 < 17.0, px60
assert px100 < 10.5, px100

print('\n>>> 主摄下 60 m 外只有 %.1f px，100 m 外只有 %.1f px。' % (px60, px100))
for s in (8, 16, 32):
    print('    在 stride %-2d 的特征图上，60 m 的标志只占 %.2f x %.2f 个格子。'
          % (s, px60 / s, px60 / s))
print('⚠️  stride 32 上它连一个格子都占不满 —— 下采样已经把它抹掉了（C57 模块 01）。')

In [ ]:
def iou_shift(size, d):
    '''两个同尺寸正方形框，在 x 与 y 方向**各**偏移 d 像素时的 IoU。'''
    ov = max(0.0, size - d)
    inter = ov * ov
    union = 2.0 * size * size - inter
    return inter / union

print('同一个 IoU 阈值对不同尺度是**极不公平**的：')
print('%10s %10s %10s %10s' % ('框边长', 'd=1px', 'd=2px', 'd=3px'))
for s in (8, 16, 32, 64):
    print('%10d %10.3f %10.3f %10.3f' % (s, iou_shift(s, 1), iou_shift(s, 2), iou_shift(s, 3)))

assert abs(iou_shift(8, 2) - 36.0 / 92.0) < 1e-9
assert iou_shift(64, 2) > 0.88
assert iou_shift(8, 2) < 0.5 < iou_shift(64, 2)
print('\n>>> 8x8 的框只要各偏 2 px，IoU 就从 1.00 掉到 %.2f；' % iou_shift(8, 2))
print('    而 64x64 的框同样偏 2 px，IoU 还有 %.2f。' % iou_shift(64, 2))
print('⚠️  于是用 IoU>0.5 做正样本分配时，**小目标几乎匹配不到任何 anchor** ——')
print('    这是模块 02（标签分配）与 C57（小目标）共同的出发点。')

## ✏️ 练习 1：帧预算分解器

实现 `frame_budget(fps, fixed_ms, task_share)`，返回
`(帧周期 ms, 感知总时间窗 ms, 该任务预算 ms)`。

规则：帧周期 = 1000/fps；感知窗口 = 帧周期 − fixed_ms，**若为负则取 0**
（固定开销已经吃光帧周期，感知一点时间都没有）；任务预算 = 感知窗口 × task_share。

In [ ]:
def frame_budget(fps, fixed_ms, task_share):
    # TODO: 返回 (frame_ms, perception_ms, task_ms)，注意 perception_ms 不能为负
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
fm, pm, tb = frame_budget(30, 17.0, 0.15)
assert abs(fm - 1000 / 30) < 1e-9, fm
assert abs(pm - (1000 / 30 - 17.0)) < 1e-9, pm
assert abs(tb - pm * 0.15) < 1e-9, tb
assert abs(tb - tsr_budget) < 1e-9, '应当复现正文里的 2.45 ms'

fm2, pm2, tb2 = frame_budget(10, 17.0, 0.15)
assert abs(fm2 - 100.0) < 1e-9 and pm2 > pm and tb2 > tb, '降帧率 -> 单帧预算变宽'

fm3, pm3, tb3 = frame_budget(60, 20.0, 0.15)
assert pm3 == 0.0 and tb3 == 0.0, '60 FPS 帧周期只有 16.7 ms，20 ms 固定开销已吃光'

print('%6s %10s %14s %14s' % ('FPS', '固定开销', '感知窗口(ms)', 'TSR 预算(ms)'))
for fps, fx in [(10, 17.0), (15, 17.0), (30, 17.0), (30, 25.0), (60, 20.0)]:
    a, b, c = frame_budget(fps, fx, 0.15)
    print('%6d %10.1f %14.2f %14.2f' % (fps, fx, b, c))
print('\n✅ 练习 1 通过。注意第 4 行：固定开销从 17 涨到 25 ms，TSR 预算直接腰斩还不止 ——')
print('   **感知预算对固定开销的敏感度是非线性的**，这是排期谈判里最有用的一条杠杆。')

## ✏️ 练习 2：帕累托前沿与「谁支配了它」

实现 `pareto_and_dominators(models)`，返回
`(前沿模型名列表（按延迟升序）, {被支配模型名: [支配它的模型名, ...]})`。

注意：**完全相同的两个点互不支配**（没有任何一项严格更好），两个都在前沿上。
实现时请用**下标**而不是 `is not` 来排除自身，否则重复元组会出错。

In [ ]:
def pareto_and_dominators(models):
    # TODO: 返回 (front_names, dom_map)
    #   front_names: 不被任何其他模型支配的模型名，按延迟升序
    #   dom_map:     只包含被支配的模型；值是支配它的模型名列表
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
front_names, dom = pareto_and_dominators(MODELS)
assert front_names == ['YOLOv5-S', 'YOLOv8-S', 'RT-DETR-R18', 'YOLOv8-M',
                       'RT-DETR-R50', 'RT-DETR-R101'], front_names
assert set(dom['YOLOv5-L']) == {'YOLOv8-M', 'RT-DETR-R50'}, dom['YOLOv5-L']
assert dom['RTMDet-S'] == ['YOLOv8-S'], dom['RTMDet-S']
assert all(n not in dom for n in front_names), '前沿上的模型不该出现在被支配表里'
assert len(dom) == len(MODELS) - len(front_names)

tie = [('A', 1.0, 10.0), ('B', 1.0, 10.0)]
assert pareto_and_dominators(tie)[0] == ['A', 'B'], '完全相同的两点互不支配'
assert pareto_and_dominators(tie)[1] == {}

print('前沿:', ' -> '.join(front_names))
print('被支配:', ', '.join('%s(被%d个)' % (k, len(v)) for k, v in dom.items()))
print('\n✅ 练习 2 通过：这就是模块 05 选型脚本的内核。')

## ✏️ 练习 3：从「要在多远检出」反推相机配置

实现 `max_detect_distance(sign_m, min_px, width_px, hfov_deg)`：
给定检测器能稳定工作的**最小像素边长** `min_px`，返回这块标志能被检出的最远距离（米）。

这是 TSR 系统设计里最先要算的一笔账：**「我要在 80 m 外看到限速牌」直接决定了
相机分辨率、FOV，以及检测器的输入分辨率与最细 stride**（C57 模块 05 会展开）。

In [ ]:
def max_detect_distance(sign_m, min_px, width_px, hfov_deg):
    # TODO: 用 focal_px 求焦距，再由 px = f*S/Z 反解 Z
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
d = max_detect_distance(0.6, 16, 1920, 60.0)
assert 62.0 < d < 63.0, d

d2 = max_detect_distance(0.6, 16, 3840, 60.0)
assert abs(d2 - 2 * d) < 1e-6, '横向分辨率翻倍 -> 焦距翻倍 -> 检出距离翻倍'

d3 = max_detect_distance(0.6, 16, 1920, 30.0)
assert d3 > 2 * d, 'FOV 减半带来的焦距增益超过 2 倍（tan 是非线性的）'

assert max_detect_distance(1.2, 16, 1920, 60.0) > d, '大牌子看得更远'
assert max_detect_distance(0.6, 8, 1920, 60.0) > d, '模型能吃更小的目标 -> 看得更远'

print('%-34s %14s' % ('配置（min_px=16, 0.6m 限速牌）', '最远检出距离'))
for label, args in [('1920 / 60° 主摄',      (0.6, 16, 1920, 60.0)),
                    ('3840 / 60° 高分主摄',  (0.6, 16, 3840, 60.0)),
                    ('1920 / 30° 长焦',      (0.6, 16, 1920, 30.0)),
                    ('1920 / 60°, min_px=8', (0.6,  8, 1920, 60.0))]:
    print('%-34s %11.1f m' % (label, max_detect_distance(*args)))
print('\n✅ 练习 3 通过。三条可换的杠杆：**加分辨率 / 减 FOV / 让模型吃更小的目标**。')
print('   前两条要动硬件（几年前就定死了），只有第三条是算法能做的 —— 这就是 C57 的全部意义。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def frame_budget(fps, fixed_ms, task_share):
    frame_ms = 1000.0 / fps
    perception_ms = max(0.0, frame_ms - fixed_ms)
    return frame_ms, perception_ms, perception_ms * task_share

In [ ]:
# 练习 2 参考答案
def pareto_and_dominators(models):
    front, dom = [], {}
    for i, m in enumerate(models):
        ds = [o[0] for j, o in enumerate(models) if j != i and dominates(o, m)]
        if ds:
            dom[m[0]] = ds
        else:
            front.append(m)
    return [m[0] for m in sorted(front, key=lambda x: x[1])], dom

In [ ]:
# 练习 3 参考答案
def max_detect_distance(sign_m, min_px, width_px, hfov_deg):
    f = focal_px(width_px, hfov_deg)
    return f * sign_m / min_px

---
## 🧪 真实工程胶囊：一份可直接抄走的延迟测量协议

论文的 FPS 不可信（模块 05 会拆穿这件事）。下面这段是**你自己重测时必须遵守的协议**，
可以原样复制到有 GPU 的机器上。

In [ ]:
RECIPE = r'''
# ============ 实时检测器延迟测量协议（照抄即可） ============
# 原则：报 p99 不报均值；含预处理与 NMS；固定硬件/精度/batch；同一协议测所有候选。

import time, numpy as np

WARMUP, RUNS = 50, 500          # ① warmup 必须有：首次 kernel 选择/显存分配会污染前几次
BATCH = 1                       # ② 车端就是 batch=1。用 batch=32 测出来的 FPS 与你无关

lat = []
for i in range(WARMUP + RUNS):
    torch.cuda.synchronize()    # ③ **必须同步**，否则你测的是 kernel launch 的时间
    t0 = time.perf_counter()

    blob = preprocess(frame)        # ④ 含预处理！论文常常不含
    d_in.copy_(blob)                #    含 H2D 拷贝
    out = engine.infer(d_in)        #    网络前向
    boxes = decode(out)             #    解码
    boxes = nms(boxes, iou=0.65)    # ⑤ **含 NMS**！这是延迟方差的唯一来源
    boxes = to_image_coords(boxes)  #    letterbox 逆变换

    torch.cuda.synchronize()
    if i >= WARMUP:
        lat.append((time.perf_counter() - t0) * 1000)

lat = np.array(lat)
print("p50=%.2f  p90=%.2f  p99=%.2f  max=%.2f ms"
      % (np.percentile(lat,50), np.percentile(lat,90),
         np.percentile(lat,99), lat.max()))

# ⑥ **必须在多种场景下各测一遍** —— NMS 的耗时随候选框数变化
#    空旷高速 / 城市主干道 / 施工区门架 / 雨夜误检爆炸，四组各 500 次
# ⑦ 报告时必须同时给出：GPU 型号、驱动、TRT 版本、精度(fp16/int8)、
#    输入分辨率、score 阈值、iou 阈值、topk。缺一项这个数字就不可复现。

# ============ 选型判据 ============
# 通过条件： p99 <= 预算  且  在四种场景下都成立
#           （p50 达标而 p99 不达标 = 不达标）
'''
print(RECIPE)
for key in ['WARMUP', 'synchronize', 'preprocess', 'nms(', 'p99', 'BATCH = 1', 'topk']:
    assert key in RECIPE, key
print('✅ 协议覆盖：warmup / 同步 / 含预处理 / 含 NMS / batch=1 / p99 / 多场景 / 环境三元组')

### 小结

- **「实时」= 两个硬约束（延迟预算、显存）+ 一个优化目标（精度）**。
  延迟预算是外部给定的，不能靠「模型更准」去谈判。
- **33.3 ms 的帧周期里，TSR 只分到约 2.45 ms**；这 2.45 ms 里网络前向只占 60%。
  **「换个更快的 backbone」对端到端最多带来 30% 的收益。**
- **NMS 是唯一随场景波动的一项**（O(n²)）。top-k 预筛能封住最坏情况，但消不掉方差。
  **车端安全关心的是 p99，而论文只报均值。**
- **选型第一步是划掉被支配的模型**（本例 11 个里划掉 5 个），第二步是
  **同延迟比 AP**——因为预算是硬约束、AP 是可争取量。跨预算档比 AP 没有选型价值。
- **TSR = 极小目标 + 极紧预算**：60 m 外的限速牌只有 16.6 px，在 stride 32 上占不满一个格子；
  8×8 的框各偏 2 px，IoU 就掉到 0.39 —— 这两个数字是模块 02 与 C57 的共同出发点。

下一站：**模块 01 · YOLO 家族演进** —— 把「每一代到底改了什么」拆成可归因的三条主线。